# Inverse Dynamics with PID control
- I term added

#### 1. Make scene with MjSpec

In [1]:
import os
import sys
import numpy as np
import time
import mujoco
sys.path.append(os.path.abspath('../'))
from pp_base_mujoco.VIEWER import *
from pp_base_mujoco.UTILS import *
from pp_base_mujoco.SPEC_HELPER import *
from pp_base_mujoco.KINEMATICS import *

In [2]:
spec_helper = MjSpecHelper()
spec_helper.add_robot(
    path='../asset/panda/panda_ee_sphere.xml',
    body_name="base",
    p=(0, 0.7, 0),
    r=(0, 0, -1.57),
    # r=(0, 0, 0),
    prefix="",
    suffix=""
)
spec_helper.add_body(
    name="box",
    freejoint = False,
    p=(0.2, 0, 0.15),
    r=(0, 0, 0),
)
model, data = spec_helper.compile()
spec_helper.save_to_xml("../asset/xml/scene_panda_lr.xml")

In [3]:
# control frequency: 1000Hz
model.opt.timestep = 0.001
dt = model.opt.timestep

In [4]:
joint_names = get_joint_names(model, data)
actuator_names = get_actuator_names(model, data)

""" LOAD TRAJECTORY """
# q_traj_left = np.load("./traj/q_left_traj_20.npy")
q_traj_left = np.load("./traj/q_left_traj_100.npy")
print(f"q_traj_left shape: {q_traj_left.shape}")

""" GO TO INITIAL QPOS """
qpos_init = q_traj_left[0]
apply_qpos_names(model, data, names=joint_names, value=qpos_init)
mujoco.mj_forward(model, data)

""" VIEWER """
viewer = MUJOCOGLVIEWER(model, data)
viewer.view_geom(group=0, show=False)
viewer.view_geom(group=1, show=True)
mujoco.mj_resetData(model, data)
apply_qpos_names(model, data, names=joint_names, value=qpos_init)
mujoco.mj_kinematics(model, data)

while viewer.is_alive():
    mujoco.mj_kinematics(model, data)
    viewer.render()

viewer.close()
del(viewer)

q_traj_left shape: (100, 7)


#### 2. Util function: get contact of certain body

In [5]:
def get_contact_body_force_position(
        model,
        data,
        body1_name,
        body2_name,
        ):
    body1_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, body1_name)
    body2_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, body2_name)
    contact_forces = []
    contact_positions = []
    for i in range(data.ncon):
        contact = data.contact[i]
        body1_in_contact_id = model.geom_bodyid[contact.geom1]
        body2_in_contact_id = model.geom_bodyid[contact.geom2]
        if (body1_in_contact_id == body1_id and body2_in_contact_id == body2_id) or (body1_in_contact_id == body2_id and body2_in_contact_id == body1_id):
            force = np.zeros(6) # (6,)
            mujoco.mj_contactForce(model,data,i,force)
            contact_forces.append(force)
            contact_positions.append(contact.pos)
    return contact_forces, contact_positions

#### 3. Inverse Dynamics
1. Calculate qacc from desired qpos and qvel (PD Control)
2. Feed desired qacc to mujoco, with mj_inverse
3. Feed torque

In [6]:
Kp = 10.0
Kd = 2.0
Ki = 0.1 

In [7]:
q_error_sum = np.zeros_like(q_traj_left[0])

qpos_goal = q_traj_left[1]
qvel_goal = np.zeros_like(qpos_goal)
qpos_diff = qpos_goal - data.qpos
q_error_sum += qpos_diff * dt
qvel_diff = qvel_goal - data.qvel

qacc_desired = Kp * qpos_diff + Kd * qvel_diff + Ki * q_error_sum
data.qacc[:] = qacc_desired
mujoco.mj_inverse(model, data)
torque = data.qfrc_inverse
print(f"torque: {torque}")

torque: [ 8.05897923e-02 -2.52908969e+01 -1.08930549e+01  8.17947722e+00
  3.43240827e-01 -7.82376074e-01 -1.19471999e-03]


#### 4. Loop

In [8]:
""" VIEWER """
viewer = MUJOCOGLVIEWER(model, data)
viewer.view_geom(group=0, show=False)
viewer.view_geom(group=1, show=True)
mujoco.mj_resetData(model, data)
apply_qpos_names(model, data, names=joint_names, value=q_traj_left[0])
mujoco.mj_kinematics(model, data)

idx = 0
render_tick = 0
q_error_sum = np.zeros_like(q_traj_left[0])
while viewer.is_alive():
    q__current = get_qpos_with_names(model, data, names=joint_names)
    qvel_current = get_qvel_with_names(model, data, names=joint_names)
    qpos_diff = qpos_goal - q__current
    q_error_sum += qpos_diff * dt
    qvel_diff = qvel_goal - qvel_current
    qacc_desired = Kp * qpos_diff + Kd * qvel_diff + Ki * q_error_sum
    data.qacc[:] = qacc_desired
    mujoco.mj_inverse(model, data)
    torque = data.qfrc_inverse
    apply_ctrl_names(model, data, names=actuator_names, value=torque)
    mujoco.mj_step(model, data)
    if render_tick % 20 == 0:
        viewer.render()
    render_tick += 1
    # waypoint switching 
    pos_ok = (
        np.max(np.abs(qpos_diff)) < 0.01
    )
    vel_ok = (
        np.max(np.abs(qvel_diff)) < 0.05
    )
    if pos_ok and vel_ok:
        if idx < len(q_traj_left) - 1:
            # if running_flag == True:
            idx += 1
            qpos_goal = q_traj_left[idx]
            q_error_sum = np.zeros_like(q_traj_left[0])
            print(f"\nSwitching to waypoint {idx}...")
    # print(f"\r idx: {idx}, pos diff norm: {np.linalg.norm(qpos_diff):.4f}", end="")
viewer.close()
del(viewer)


Switching to waypoint 1...

Switching to waypoint 2...

Switching to waypoint 3...

Switching to waypoint 4...

Switching to waypoint 5...

Switching to waypoint 6...

Switching to waypoint 7...

Switching to waypoint 8...

Switching to waypoint 9...

Switching to waypoint 10...

Switching to waypoint 11...

Switching to waypoint 12...

Switching to waypoint 13...

Switching to waypoint 14...

Switching to waypoint 15...

Switching to waypoint 16...

Switching to waypoint 17...

Switching to waypoint 18...

Switching to waypoint 19...

Switching to waypoint 20...

Switching to waypoint 21...

Switching to waypoint 22...

Switching to waypoint 23...

Switching to waypoint 24...

Switching to waypoint 25...

Switching to waypoint 26...

Switching to waypoint 27...

Switching to waypoint 28...

Switching to waypoint 29...

Switching to waypoint 30...

Switching to waypoint 31...

Switching to waypoint 32...

Switching to waypoint 33...

Switching to waypoint 34...

Switching to waypoint 